In [1]:
import torch, glob, random , cv2
from torchvision import models
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from data_transform import conditional_resize_pad, offset_crop, PercentileNormalize, RayleighMatchTensor, LoadUint16TIff
from augmentation import mig_aug_train
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np 
num_classes = 3
weights = "checkpoint_FBH/65_fighter_bomber_helicopter_100_32_dropout0.35_weight_decay"

In [9]:
def load_model(weights, num_classes, device = 'cuda'):
    model = models.mobilenet_v3_small(weights = None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    weights = torch.load(weights, map_location=device)
    model.load_state_dict(weights)
    model.to(device)
    model.eval()
    return model


def preprocess_image(image_path, image_type = 'real', gamma = 0.4, minp = 10, maxp =99, img_size = 128):
    image = image_path
    transform = None
    if image_type == 'real':
        transform = transforms.Compose([
            LoadUint16TIff(gamma=1, minp=5, maxp=98.5, img_size=128), 
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        image = Image.open(image_path).convert('RGB')
        transform = transforms.Compose([
            transforms.Lambda(lambda img: offset_crop(img)(img)),
            transforms.ToTensor(), 
            transforms.Lambda(mig_aug_train()), 
            transforms.Normalize([0.5,0.5,0.5],[0.25,0.25,0.25])
        ])

        return transform(image).unsqueeze(0)

def predict(model, image_tensor, device = "cuda", topk =2):
    image_tensor = image_tensor.to(device) 
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim =1)
        top_probs, top_labels = torch.topk(probs, topk)
        return top_probs.cpu().numpy()[0], top_labels.cpu().numpy()[0]

def inference(weights, num_classes, image_path, device = "cuda", topk =2, image_type = 'real'):
    import os 
    labels = os.path.dirname(os.path.dirname(image_path))
    labels = sorted(os.listdir(labels))
    
    model = load_model(weights=weights, num_classes=num_classes, device=device)
    image_tensor = preprocess_image(image_path, image_type)
    prob, label = predict(model, image_tensor)
    print("predicted", labels[label[0]], "actual", os.path.dirname(image_path).split('/')[-1], image_path.split('/')[-1])
    #print("actual", os.path.dirname(image_path).split('/')[-1])


In [10]:
model  = load_model(weights, num_classes)

In [28]:
test_image = "bomber_figther_helicopter/train/fighter/All_Reflections_Fr0_deg_.png"
heli = "bomber_figther_helicopter/train/helicopter/mi28_3sc_1pix_wings_0.9ham_310.png"
image_tensor  = preprocess_image(heli, "fake")

In [29]:
model(image_tensor.to("cuda"))

tensor([[-1.2371, -2.2585,  3.9320]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

In [30]:
range(1)

range(0, 1)

In [ ]:
# target_layers = [model.features[-1]]
# cam = GradCAM(model=model, target_layers=target_layers)

# targets = [ClassifierOutputTarget(0)]
# grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
# visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

SyntaxError: invalid syntax (1068564517.py, line 1)

In [24]:
import glob
image_dir = '/media/sphere/744c0eb8-a6d5-469f-9d27-449a9eef9aa1/home/admin123/Kanishk/classification/TU95Classification/val/Chinook'
images = glob.glob(image_dir + '/*')
weights = "/media/sphere/744c0eb8-a6d5-469f-9d27-449a9eef9aa1/home/admin123/Kanishk/classification/checkpoint/75_Bomber_vs_helicopter_subparts_mblv3_val_g0.4_freeze_bbone_75acc.pth"

for name in images:
    inference(weights, 2, name)

predicted Chinook actual Chinook Ch003.tif
predicted Chinook actual Chinook H009.tif
predicted TU_95 actual Chinook H007.tif
predicted TU_95 actual Chinook Ch008.tif
predicted Chinook actual Chinook H002.tif
predicted TU_95 actual Chinook H006.tif
predicted Chinook actual Chinook H010.tif
predicted Chinook actual Chinook H004.tif
predicted Chinook actual Chinook Ch007.tif
predicted TU_95 actual Chinook Ch006.tif
predicted Chinook actual Chinook H003.tif
predicted Chinook actual Chinook H005.tif
predicted Chinook actual Chinook Ch004.tif
predicted Chinook actual Chinook Ch009.tif
predicted Chinook actual Chinook Ch005.tif
predicted Chinook actual Chinook Ch002.tif
predicted TU_95 actual Chinook H008.tif
predicted TU_95 actual Chinook Ch001.tif
predicted Chinook actual Chinook H001.tif


0.8421052631578947

In [1]:
import rasterio

In [3]:
from osgeo import gdal as gd 

In [8]:
image = gd.Open('/media/sphere/744c0eb8-a6d5-469f-9d27-449a9eef9aa1/home/admin123/Kanishk/Transport/t2/SkyFi_2442S14A-1_2024-10-17_0305Z_SAR_VERY-HIGH_Qinghai-China.tif')

In [9]:
image.GetProjection()

'GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'